In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Sample data
tour_descriptions = [
    "A beautiful wildlife safari in Rajasthan with tigers and leopards.",
    "A historical fort tour with guided sightseeing.",
    "An adventure jungle trek with various bird species and tigers."
]

# Convert text into TF-IDF vectors
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(tour_descriptions)

# Compute cosine similarity
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(cosine_sim)  # Output: Similarity matrix


[[1.         0.05505387 0.18843891]
 [0.05505387 1.         0.05171638]
 [0.18843891 0.05171638 1.        ]]


In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load tour data from CSV
df = pd.read_csv("rawTourData.csv")

# Check the structure of the dataset
print(df.head())  # Ensure the 'description' column exists

# Fill missing values with empty strings
df['description'] = df['description'].fillna("")

# Convert text into TF-IDF vectors
vectorizer = TfidfVectorizer(stop_words='english')  # Removes common words like 'the', 'and'
tfidf_matrix = vectorizer.fit_transform(df['description'])

# Compute cosine similarity matrix
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

# Print the similarity matrix
print(cosine_sim)


                     tour_name  \
0     Ranthambore Tiger Safari   
1      Corbett Tiger Adventure   
2  Bandhavgarh Wilderness Tour   
3         Kanha Jungle Retreat   
4       Tadoba Wildlife Safari   

                                         description      best_time  \
0  Ranthambore National Park, located in Rajastha...   October-June   
1  Jim Corbett National Park, nestled in the Hima...  November-June   
2  Bandhavgarh National Park in Madhya Pradesh is...   October-June   
3  Kanha National Park, located in Madhya Pradesh...   October-June   
4  Tadoba-Andhari Tiger Reserve in Maharashtra is...   October-June   

                       location tour_type  
0       Ranthambore (Rajasthan)  Wildlife  
1         Corbett (Uttarakhand)  Wildlife  
2  Bandhavgarh (Madhya Pradesh)  Wildlife  
3        Kanha (Madhya Pradesh)  Wildlife  
4          Tadoba (Maharashtra)  Wildlife  
[[1.         0.19303711 0.24860003 ... 0.01488703 0.         0.        ]
 [0.19303711 1.         0.22123

In [3]:
def recommend_tours(tour_name, df, cosine_sim, top_n=5):
    # Get the index of the tour that matches the tour_name
    idx = df[df['tour_name'] == tour_name].index[0]

    # Get the similarity scores for all tours with that tour
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort the tours based on similarity scores (descending order)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get the indices of the top_n most similar tours (excluding itself)
    top_tour_indices = [i[0] for i in sim_scores[1:top_n+1]]

    # Get the tour names of the recommended tours
    recommended_tours = df.iloc[top_tour_indices][['tour_name', 'location', 'tour_type']]

    return recommended_tours


In [6]:
# Get recommendations for "Ranthambore Tiger Safari"
recommended = recommend_tours("Ranthambore Tiger Safari", df, cosine_sim, top_n=10)

# Display recommendations
print(recommended)


                            tour_name                            location  \
54        Ranthambore Wildlife Safari             Ranthambore (Rajasthan)   
243  Ranthambore Fort and Museum Tour                           Rajasthan   
196  Ranthambore National Park Safari                           Rajasthan   
194  Jim Corbett National Park Safari                         Uttarakhand   
9              Nagarhole Tiger Trails               Nagarhole (Karnataka)   
2         Bandhavgarh Wilderness Tour        Bandhavgarh (Madhya Pradesh)   
6            Pench Jungle Exploration  Pench (Madhya Pradesh/Maharashtra)   
8              Satpura Wild Adventure            Satpura (Madhya Pradesh)   
199        Kanha National Park Safari                      Madhya Pradesh   
4              Tadoba Wildlife Safari                Tadoba (Maharashtra)   

                 tour_type  
54               Adventure  
243  Monuments and Museums  
196                 Nature  
194                 Nature  
9      

In [ ]:
# Get recommendations for "beach"
recommended = recommend_tours("beach", df, cosine_sim, top_n=3)

# Display recommendations
print(recommended)

#########################################################################
############  Here, it will give error, because we are passing a random string to the function instead of 
############  a real name of a tour that can be found in the tour-Name column of the dataset
############  that is why this approach is not good.

IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
############################################################
############       GOOD APPROACH        ####################


from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Load dataset (Assuming 'rawTourData.csv' is already loaded into a DataFrame)
df = pd.read_csv("rawTourData.csv")

# Combine 'tour_name' and 'description' into a single text field
df['combined_text'] = df['tour_name'] + " " + df['description']

# Create a TF-IDF vectorizer
vectorizer = TfidfVectorizer(stop_words='english')

# Convert the tour descriptions into numerical vectors
tfidf_matrix = vectorizer.fit_transform(df['combined_text'])

def search_tours(user_query, df, tfidf_matrix, top_n=5):
    """
    Search for the most relevant tours based on user input.
    """
    # Convert the user query into a TF-IDF vector
    user_query_vector = vectorizer.transform([user_query])

    # Compute cosine similarity between user query and all tour descriptions
    cosine_sim = cosine_similarity(user_query_vector, tfidf_matrix)

    # Get similarity scores and sort in descending order
    sim_scores = list(enumerate(cosine_sim[0]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get indices of the top N matching tours
    top_tour_indices = [i[0] for i in sim_scores[:top_n]]

    # Return top matching tours
    return df.iloc[top_tour_indices][['tour_name', 'location', 'tour_type', 'description']]



In [12]:
# Example: Search for "beach" related tours
user_input = "beach"
recommended_tours = search_tours(user_input, df, tfidf_matrix, top_n=5)

# Display recommendations
print(recommended_tours)


                                  tour_name         location  \
175                        Dumas Beach Tour          Gujarat   
76                        Goa Beach Holiday        Goa (Goa)   
343  South Goa Spring Beach Relaxation Tour   South Goa, Goa   
293                       Chennai Food Tour       Tamil Nadu   
323   Andaman Islands Summer Beach Vacation  Andaman Islands   

                tour_type                                        description  
175       Mystery & Scary  Dumas Beach, located in Gujarat, is infamous f...  
76             Relaxation  Goa is India's ultimate beach destination, kno...  
343      Seasonal Tourism  Enjoy the sunny and pleasant weather of South ...  
293  Food Culture Tourism  Taste the authentic South Indian cuisine, incl...  
323      Seasonal Tourism  Relax on the pristine beaches of the Andaman I...  


In [ ]:
# Example: Search for "fun" related tours
user_input = "fun"
recommended_tours = search_tours(user_input, df, tfidf_matrix, top_n=15)

# Display recommendations
print(recommended_tours)


                         tour_name                            location  \
79        Jaipur Cultural Fun Tour                  Jaipur (Rajasthan)   
86     Dubai-Inspired Fun in Delhi                       Delhi (Delhi)   
82    Rishikesh Adventure and Yoga             Rishikesh (Uttarakhand)   
78           Manali Snow Adventure           Manali (Himachal Pradesh)   
84  Kashmir Valley Family Vacation           Kashmir (Jammu & Kashmir)   
81    Kochi Backwaters and Beaches                      Kochi (Kerala)   
83      Alleppey Houseboat Holiday                   Alleppey (Kerala)   
76               Goa Beach Holiday                           Goa (Goa)   
0         Ranthambore Tiger Safari             Ranthambore (Rajasthan)   
1          Corbett Tiger Adventure               Corbett (Uttarakhand)   
2      Bandhavgarh Wilderness Tour        Bandhavgarh (Madhya Pradesh)   
3             Kanha Jungle Retreat              Kanha (Madhya Pradesh)   
4           Tadoba Wildlife Safari    